# YSSY Wind Forecast — Complete Workflow

**Source folder:** `team-members/kecheng-zhang/scripts/YSSY-code/`

This notebook consolidates every script in that folder into a single end-to-end pipeline:

```
Raw BOM fixed-width files  (RawDataC/)
        │
        ▼  data-processing/process_data.py  [STAGE 0]
  ProcessedData/*.txt  (parsed CSV, one per station)
        │
        ▼  data-processing/merge_stations.py
  Station merge (older + newer instrument files)
        │
        ▼  data-processing/plot_availability.py.py  [DIAGNOSTIC A]
  Availability plot (raw)
        │
        ▼  data-processing/interpolateData0.5.py
  Single-gap fill (30 min)
        │
        ▼  data-processing/outage_lengths.py  [DIAGNOSTIC B]
  Outage distribution plots
        │
        ▼  data-processing/convertWindComponents.py
  Wind speed / dir → U / V
        │
        ▼  data-processing/interpolate.py
  Spline fill (gaps ≤ 2.5 hr)
        │
        ▼  data-processing/plotTotalAvailability.py  [DIAGNOSTIC D]
  Per-element + concurrent 24H completeness (post-spline)
        │
        ▼  data-processing/drop_columns.py
  Drop unwanted columns
        │
        ▼  data-processing/crop_to_years.py
  Crop to [START_YEAR, END_YEAR]
        │
        ▼  data-processing/find_missing_timestamp.py  [DIAGNOSTIC C]
  Missing-timestamp check
        │
        ▼  ★ PARQUET SAVE ★
  parquet/yssy/
        │
        ▼  data-processing/preprocess_data.py
  Feature engineering + train / val / test split  →  parquet/yssy/ml/
        │
        ├─▶  model/YSSY_winds_24hr.py        [Option A — quick fixed-param model]
        ├─▶  model/YSSY_LightGMB.py          [Option B — Optuna multi-output]
        └─▶  model/train_final_individual.py [Option C — per-target individual]
                    ↑ tuned by model/tune_single_model.py
        │
        ▼  model/evaluate_all_24hr_models.py
  Evaluation (MAE / MSE per horizon)
        │
        ▼  model/plot_samples_24hr.py  +  model/plot_test_samples.py
  Forecast visualisation
```

Each section header includes the source file name.

---
## Stage 0 — Raw BOM Data Ingestion
**Source:** `data-processing/process_data.py`

Reads raw Bureau of Meteorology **fixed-width** `.txt` files from `RawDataC/` and writes
clean CSV files to `ProcessedData/` (one file per station).

Key steps (from `process_data.py → process_weather_file()`):
- Read with `pd.read_fwf` using BOM byte-offset col_specs
- Build `timestamp` from year / month / day / hour / minute fields
- Convert measurement columns to numeric (coerce errors → NaN)
- Create `data_completeness` flag (1 if all core fields are non-NaN)
- Save as `{station_id}.txt` (CSV format)

> **Note:** This stage requires the raw BOM files in `RawDataC/` (not committed to the repo).  
> If `ProcessedData/*.txt` already exist, skip this cell and proceed to Stage 1.

In [ ]:
# SOURCE: data-processing/process_data.py

_COL_SPECS = [
    (0, 2),   (3, 9),   (10, 14), (15, 17), (18, 20), (21, 23), (24, 26),
    (27, 32), (33, 34), (35, 40), (41, 42), (43, 48), (49, 50),
    (51, 54), (55, 56), (57, 62), (63, 64), (65, 66), (67, 68),
    (69, 70), (71, 72), (73, 74), (75, 76), (77, 78), (79, 80),
    (81, 87), (88, 89), (90, 92), (93, 94),
]
_COL_NAMES = [
    'record_id', 'station_id',
    'year', 'month', 'day', 'hour', 'minute',
    'air_temp', 'q_air_temp',
    'dew_point', 'q_dew_point',
    'wind_speed', 'q_wind_speed',
    'wind_dir', 'q_wind_dir',
    'max_gust_speed', 'q_max_gust_speed',
    'cloud1_amt', 'q_cloud1_amt',
    'cloud2_amt', 'q_cloud2_amt',
    'cloud3_amt', 'q_cloud3_amt',
    'cloud4_amt', 'q_cloud4_amt',
    'msl_pressure', 'q_msl_pressure',
    'aws_flag', 'end_indicator',
]
_NUMERIC_COLS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir',
                 'max_gust_speed', 'msl_pressure', 'aws_flag']
_FINAL_COLS   = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']


def ingest_raw_bom_file(filepath, processed_dir):
    """
    Parse one raw BOM fixed-width file and save to processed_dir.
    Source: data-processing/process_data.py -> process_weather_file()
    """
    filepath, processed_dir = Path(filepath), Path(processed_dir)
    df = pd.read_fwf(filepath, colspecs=_COL_SPECS, names=_COL_NAMES, dtype=str, skiprows=1)
    if df.empty:
        print(f'  {filepath.name}: empty, skipped.')
        return

    sid = df['station_id'].dropna().iloc[0].strip()
    df['timestamp_str'] = (
        df['year'].str.strip()  + '-' + df['month'].str.strip() + '-' +
        df['day'].str.strip()   + ' ' + df['hour'].str.strip()  + ':' +
        df['minute'].str.strip()
    )
    df['timestamp'] = pd.to_datetime(df['timestamp_str'], format='%Y-%m-%d %H:%M', errors='coerce')

    for col in _NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')

    df['data_completeness'] = (
        df[_FINAL_COLS].notna().all(axis=1) & df['timestamp'].notna()
    ).astype(int)
    out = df[['timestamp'] + _FINAL_COLS + ['data_completeness']]
    out.to_csv(processed_dir / f'{sid}.txt', index=False, na_rep='NaN')
    pct = out['data_completeness'].mean() * 100
    print(f'  {filepath.name} -> {sid}.txt  ({len(out):,} rows, {pct:.1f}% complete)')


RAW_DATA_DIR = YSSY_CODE_DIR / 'RawDataC'
if RAW_DATA_DIR.exists():
    raw_files = sorted(RAW_DATA_DIR.glob('*Data*.txt'))
    print(f'Found {len(raw_files)} raw file(s) in {RAW_DATA_DIR}')
    for rf in raw_files:
        ingest_raw_bom_file(rf, RAW_TXT_DIR)
    print('Raw data ingestion complete.')
else:
    print(f'RawDataC/ not found at {RAW_DATA_DIR.resolve()}')
    print('Skipping Stage 0 — assuming ProcessedData/*.txt files already exist.')
    print(f'Stage 1 will load from: {RAW_TXT_DIR.resolve()}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.interpolate import UnivariateSpline
import joblib
import random
import glob
import time
from pathlib import Path
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

print(f'pandas {pd.__version__}  |  lightgbm {lgb.__version__}')
print('All imports OK.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  PATHS — adjust these to match your local setup
# ─────────────────────────────────────────────────────────────────────────────

# Root of kecheng's YSSY-code folder (source of all original scripts)
YSSY_CODE_DIR = Path('../../kecheng-zhang/scripts/YSSY-code')

# Raw cleaned station .txt files (CSV format, one file per station)
RAW_TXT_DIR = YSSY_CODE_DIR / 'ProcessedData'

# Outputs land in your notebooks folder
NOTEBOOK_DIR  = Path('.')
PARQUET_DIR   = NOTEBOOK_DIR / 'parquet' / 'yssy'
MODEL_DIR     = NOTEBOOK_DIR / 'models'
PLOTS_DIR     = NOTEBOOK_DIR / 'plots' / 'yssy'

for d in [PARQUET_DIR, MODEL_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────
WEATHER_PARAMS       = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir',
                         'max_gust_speed', 'msl_pressure', 'aws_flag']
INTERPOLATE_ELEMENTS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir',
                         'max_gust_speed', 'msl_pressure']
SPLINE_ELEMENTS      = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
COLUMNS_TO_DROP      = ['max_gust_speed', 'aws_flag', 'wind_dir_recalc']

# Station pairs to merge: (older_file, newer_file, output_station_id)
# Newer file takes priority; gaps filled from older file.
MERGE_PAIRS = [('YSRI Old.txt', 'YSRI.txt', 'YSRI')]

# ─────────────────────────────────────────────────────────────────────────────
#  PIPELINE PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
START_YEAR    = 2000   # crop_to_years.py: was 2000–2000 (bug: end year = start year)
END_YEAR      = 2024   # FIXED: set to intended final year
MAX_GAP_STEPS = 5      # interpolate.py: MAX_GAP_STEPS_FOR_SPLINE
SPLINE_ORDER  = 3      # interpolate.py: SPLINE_ORDER
SPLINE_SMOOTH = 1      # interpolate.py: SPLINE_SMOOTHING_FACTOR

# ─────────────────────────────────────────────────────────────────────────────
#  ML PARAMETERS  (from data-processing/preprocess_data.py)
# ─────────────────────────────────────────────────────────────────────────────
TARGET_STATION       = 'YSSY'
STATIONS             = ['BELL', 'MTB', 'YBTH', 'YCNK', 'YSBK', 'YSCN', 'YSNW', 'YSRI', 'YSSY', 'YSWG']
STATIONS_NO_PRESSURE = ['BELL', 'MTB']   # no msl_pressure sensor
ALL_FILE_FEATURES    = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
LOOKBACK_STEPS       = 48   # 24 hr history at 30-min resolution
FORECAST_STEPS       = 48   # 24 hr forecast at 30-min intervals
TRAIN_PROP, VAL_PROP, TEST_PROP = 0.8, 0.1, 0.1
TIMESTAMP_COL        = 'timestamp_t'

# ─────────────────────────────────────────────────────────────────────────────
#  LIGHTGBM HYPERPARAMETERS  (tuned values from YSSY_LightGMB.py)
# ─────────────────────────────────────────────────────────────────────────────
LGBM_PARAMS = {
    'objective':         'regression_l2',
    'metric':            'l2',
    'n_estimators':      800,
    'learning_rate':     0.02130325383067257,
    'num_leaves':        200,
    'max_depth':         17,
    'min_child_samples': 45,
    'subsample':         0.8767980400385894,
    'colsample_bytree':  0.6947769061520032,
    'reg_alpha':         1.959826832632918,
    'reg_lambda':        0.12507898430314643,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

print('Configuration loaded.')
if not RAW_TXT_DIR.exists():
    print(f'WARNING: {RAW_TXT_DIR.resolve()} not found — update RAW_TXT_DIR above.')
else:
    txts = list(RAW_TXT_DIR.glob('*.txt'))
    print(f'Found {len(txts)} station file(s) in {RAW_TXT_DIR.resolve()}')

---
## Stage 1 — Station Merge
**Source:** `data-processing/merge_stations.py`

When a station was relocated, two `.txt` files exist for the same station ID (old vs new
instrument).  This step performs an **outer merge** on timestamp; the newer file's values
take priority and gaps are filled from the older file.

> **Known issue in original:** hardcoded to exactly `YSRI.txt` + `YSRI Old.txt`.  
> Here it is generalised via the `MERGE_PAIRS` list in the config cell.

In [ ]:
# SOURCE: data-processing/merge_stations.py

def load_station_txt(filepath):
    """Read one cleaned station .txt file (CSV with 'timestamp' column)."""
    df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'],
                     infer_datetime_format=True)
    for col in WEATHER_PARAMS:
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def merge_station_pair(file1_path, file2_path):
    """
    Outer-merge two station files.  File2 (newer) values take priority.
    Source: merge_stations.py → merge_weather_data()
    """
    df1 = load_station_txt(file1_path)
    df2 = load_station_txt(file2_path)
    merged = pd.merge(df1, df2, on='timestamp', how='outer', suffixes=('_f1', '_f2'))

    for param in WEATHER_PARAMS:
        c1, c2 = f'{param}_f1', f'{param}_f2'
        if c2 in merged.columns and c1 in merged.columns:
            merged[param] = merged[c2].combine_first(merged[c1])
        elif c2 in merged.columns:
            merged[param] = merged[c2]
        elif c1 in merged.columns:
            merged[param] = merged[c1]
        else:
            merged[param] = np.nan

    merged = merged.drop(columns=[c for c in merged.columns
                                   if c.endswith('_f1') or c.endswith('_f2')])
    meas = [p for p in WEATHER_PARAMS if p != 'aws_flag' and p in merged.columns]
    merged['data_completeness'] = merged[meas].notna().all(axis=1).astype(int)
    return merged.sort_values('timestamp').reset_index(drop=True)


# ── Load all station files ────────────────────────────────────────────────────
station_files = sorted(RAW_TXT_DIR.glob('*.txt'))
older_files   = {p[0] for p in MERGE_PAIRS}

station_data = {}   # {station_id: DataFrame}  — live dict updated through all stages

for f in station_files:
    if f.name in older_files:
        continue
    sid = f.stem
    pair = next((p for p in MERGE_PAIRS if p[1] == f.name), None)
    if pair:
        df  = merge_station_pair(RAW_TXT_DIR / pair[0], f)
        sid = pair[2]
        tag = f'merged ({pair[0]} + {f.name})'
    else:
        df  = load_station_txt(f)
        tag = f.name

    df = df.set_index('timestamp').sort_index()
    station_data[sid] = df
    pct = df['data_completeness'].mean() * 100 if 'data_completeness' in df.columns else float('nan')
    print(f'  {sid:8s}: {len(df):>7,} rows  {pct:5.1f}% complete  [{tag}]')

print(f'\n{len(station_data)} stations loaded.')

---
## Diagnostic A — Data Availability Plots (raw)
**Source:** `data-processing/plot_availability.py.py`

Monthly percentage of valid (non-NaN) observations for each weather element, using
**full-hour only** records.  Run on raw merged data before any interpolation.

In [ ]:
# SOURCE: data-processing/plot_availability.py.py

AVAIL_ELEMENTS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']
AVAIL_COLORS   = plt.cm.tab10(np.linspace(0, 0.5, len(AVAIL_ELEMENTS)))

for sid, df in station_data.items():
    df_fh = df[df.index.minute == 0].copy()   # full-hour records only
    if df_fh.empty:
        print(f'{sid}: no full-hour records, skipping.')
        continue

    df_fh['year_month'] = df_fh.index.to_period('M')

    fig, ax = plt.subplots(figsize=(14, 4))
    for el, color in zip(AVAIL_ELEMENTS, AVAIL_COLORS):
        if el not in df_fh.columns:
            continue
        monthly = df_fh.groupby('year_month').agg(
            total=('air_temp', 'count'),
            avail=(el, lambda x: x.notna().sum())
        )
        monthly['pct'] = monthly['avail'] / monthly['total'].replace(0, np.nan) * 100
        ax.plot(monthly.index.to_timestamp(), monthly['pct'],
                label=el.replace('_', ' ').title(), color=color, marker='.', ms=3)

    ax.set_title(f'Station {sid} — Monthly Availability (full-hour, raw)')
    ax.set_ylabel('% available')
    ax.set_ylim(0, 105)
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'{sid}_availability_raw.png', dpi=110, bbox_inches='tight')
    plt.show()
    plt.close()

---
## Stage 2 — Single-step Gap Fill (30 min)
**Source:** `data-processing/interpolateData0.5.py`

Fills isolated **single** NaN values using the average of the two neighbours.
Gaps of 2+ consecutive NaNs are left for the spline step.

> **Known issue in original:** saves with `float_format='%.1f'` (1 decimal place), permanently
> reducing precision.  Here data stays in memory at full float64 precision.

In [ ]:
# SOURCE: data-processing/interpolateData0.5.py

def fill_single_step_gaps(series):
    """
    Linear fill of isolated single-NaN gaps (30-min gaps only).
    Source: interpolateData0.5.py → interpolate_single_step_gaps()
    """
    s    = series.copy()
    is_na = series.isna()
    for i in range(1, len(series) - 1):
        if not is_na.iloc[i-1] and is_na.iloc[i] and not is_na.iloc[i+1]:
            s.iloc[i] = (series.iloc[i-1] + series.iloc[i+1]) / 2.0
    return s


for sid, df in station_data.items():
    for el in INTERPOLATE_ELEMENTS:
        if el in df.columns:
            station_data[sid][el] = fill_single_step_gaps(df[el])

print('Single-step gap fill done.')

---
## Diagnostic B — Outage Distribution Plots
**Source:** `data-processing/outage_lengths.py`

Grouped bar charts showing how long outages (consecutive NaN runs) are for each
weather element at each station.  Run **after** the single-step fill so 30-min gaps
have already been removed.

> **Known issue in original:** uses `plt.cm.get_cmap()` which is deprecated.  Fixed here.

In [ ]:
# SOURCE: data-processing/outage_lengths.py

OUTAGE_BINS = [
    (1,  2,    '30 min'),
    (2,  3,    '1 hr'),
    (3,  5,    '1.5-2 hr'),
    (5,  13,   '2-6 hr'),
    (13, 49,   '6-24 hr'),
    (49, 337,  '1-7 day'),
    (337, 1441,'7d-1mo'),
    (1441, 17521, '1mo-1yr'),
    (17521, float('inf'), '>1 yr'),
]
BIN_LABELS = [b[2] for b in OUTAGE_BINS]
OUTAGE_ELEMENTS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']

def find_outage_durations(series):
    """Return list of consecutive-NaN run lengths."""
    outages, run = [], 0
    for na in series.isna():
        if na:
            run += 1
        elif run:
            outages.append(run); run = 0
    if run:
        outages.append(run)
    return outages

def categorise(durations, bins):
    counts = [0] * len(bins)
    for d in durations:
        for i, (lo, hi, _) in enumerate(bins):
            if lo <= d < hi:
                counts[i] += 1; break
    return counts

for sid, df in station_data.items():
    cols = [el for el in OUTAGE_ELEMENTS if el in df.columns]
    if not cols:
        continue

    x = np.arange(len(BIN_LABELS))
    w = 0.8 / len(cols)
    colors = matplotlib.colormaps['viridis'](np.linspace(0, 1, len(cols)))

    fig, ax = plt.subplots(figsize=(14, 4))
    for j, (el, c) in enumerate(zip(cols, colors)):
        counts = categorise(find_outage_durations(df[el]), OUTAGE_BINS)
        ax.bar(x + j * w - 0.4 + w/2, counts, width=w, label=el, color=c, alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(BIN_LABELS, fontsize=8)
    ax.set_title(f'{sid} — Outage Length Distribution (after 30-min fill)')
    ax.set_ylabel('Number of outages')
    ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'{sid}_outage_dist.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()

---
## Stage 3 — Wind Speed / Direction → U / V Components
**Source:** `data-processing/convertWindComponents.py`

Meteorological convention (wind **from** a direction):
```
u = -speed × sin(dir_rad)    # eastward positive
v = +speed × cos(dir_rad)    # northward positive (see note below)
```
> **Known issue in original:** the code comment says `v = -speed × cos(...)` but the code
> uses `+cos`.  This is self-consistent with `arctan2(-u, -v)` used in `interpolate.py` when
> reconstructing direction — but should be explicitly verified.

In [ ]:
# SOURCE: data-processing/convertWindComponents.py

def wind_to_uv(df):
    """
    Add u_component and v_component; drop wind_speed and wind_dir.
    Source: convertWindComponents.py → convert_to_uv_and_save()
    """
    if 'wind_speed' not in df.columns or 'wind_dir' not in df.columns:
        return df
    dir_rad = np.deg2rad(df['wind_dir'].astype(float))
    out = df.copy()
    out['u_component'] = -df['wind_speed'] * np.sin(dir_rad)
    out['v_component'] =  df['wind_speed'] * np.cos(dir_rad)
    mask = df['wind_speed'].isna() | df['wind_dir'].isna()
    out.loc[mask, ['u_component', 'v_component']] = np.nan
    return out.drop(columns=['wind_speed', 'wind_dir'])


def uv_to_wind(u, v):
    """
    Reconstruct wind speed (kt) and direction (°) from U / V.
    Source: interpolate.py (recalculation after spline fill)
    """
    u, v   = np.asarray(u, float), np.asarray(v, float)
    speed  = np.sqrt(u**2 + v**2)
    # arctan2(-u, -v) matches the +cos convention used in wind_to_uv
    dirn   = (np.rad2deg(np.arctan2(-u, -v)) + 360) % 360
    return speed, np.where(speed > 0.01, dirn, np.nan)


for sid in list(station_data):
    station_data[sid] = wind_to_uv(station_data[sid])

print('Wind → U/V conversion done.')
# Quick check: verify first station has u_component and no wind_speed
sid0 = next(iter(station_data))
print(f'  {sid0} columns: {list(station_data[sid0].columns)}')

---
## Stage 4 — Spline Gap Fill (up to 2.5 hr)
**Source:** `data-processing/interpolate.py`

Fills NaN gaps of length ≤ `MAX_GAP_STEPS` using a **cubic spline** fitted to the
`SPLINE_CONTEXT_PTS` known points on each side of the gap. After filling U/V,
`wind_speed_recalc` and `wind_dir_recalc` are derived.

Key parameters (from `interpolate.py`):
| Parameter | Value | Meaning |
|---|---|---|
| `MAX_GAP_STEPS` | 5 | Max gap steps to fill (2.5 hr) |
| `SPLINE_ORDER` | 3 | Cubic spline |
| `SPLINE_SMOOTH` | 1 | `s=1` → slight smoothing (does not pass exactly through all points) |

In [ ]:
# SOURCE: data-processing/interpolate.py

def _spline_fill_gap(series, gap_start, gap_end, ctx_pts, k=3, s=1):
    """
    Local cubic spline over a single NaN gap.
    Source: interpolate.py → cubic_spline_interpolate_gap()
    """
    n = len(series)
    before = series.iloc[max(0, gap_start - ctx_pts): gap_start].dropna()
    after  = series.iloc[gap_end + 1: min(n, gap_end + 1 + ctx_pts)].dropna()

    # Use integer positions as x-axis
    x = list(range(max(0, gap_start - ctx_pts), gap_start))[-len(before):] + \
        list(range(gap_end + 1, min(n, gap_end + 1 + ctx_pts)))[:len(after)]
    y = list(before.values) + list(after.values)

    x_u, idx = np.unique(x, return_index=True)
    y_u = np.array(y)[idx]
    if len(x_u) < k + 1:
        return None
    try:
        sp   = UnivariateSpline(x_u, y_u, k=k, s=s)
        xi   = np.arange(gap_start, gap_end + 1)
        vals = sp(xi)
        return pd.Series(vals, index=series.index[xi])
    except Exception:
        return None


def spline_fill_series(series, max_gap=MAX_GAP_STEPS, ctx=6, k=SPLINE_ORDER, s=SPLINE_SMOOTH):
    """
    Fill NaN runs of length ≤ max_gap with local cubic spline.
    Source: interpolate.py → apply_spline_interpolation_to_series()
    """
    out = series.copy()
    na  = series.isna().values
    i   = 0
    while i < len(na):
        if na[i]:
            j = i
            while j < len(na) and na[j]:
                j += 1
            if 0 < (j - i) <= max_gap:
                filled = _spline_fill_gap(series, i, j - 1, ctx, k, s)
                if filled is not None:
                    out.loc[filled.index] = filled.values
            i = j
        else:
            i += 1
    return out


print('Applying spline interpolation (may take a few minutes for 25 years of data)...')
for sid, df in station_data.items():
    for el in SPLINE_ELEMENTS:
        if el in df.columns:
            station_data[sid][el] = spline_fill_series(df[el])

    # Recalculate wind speed / direction from interpolated U / V
    if 'u_component' in station_data[sid].columns:
        spd, dirn = uv_to_wind(station_data[sid]['u_component'].values,
                                station_data[sid]['v_component'].values)
        station_data[sid]['wind_speed_recalc'] = spd
        station_data[sid]['wind_dir_recalc']   = dirn
    print(f'  {sid} done.')

print('Spline fill complete.')

### [Optional] Spline interpolation test on synthetic data
**Source:** `data-processing/interpolateTest.py`

Quick visual sanity-check: introduce a 10-step gap in a synthetic signal, fill it with the local spline, and compare to the original.

In [ ]:
# SOURCE: data-processing/interpolateTest.py  (standalone test — does not affect pipeline data)

np.random.seed(42)
n = 48
t = np.arange(n)
original = np.sin(t / ((n-1) / (2*np.pi))) + t / 20 + np.random.normal(0, 0.3, n)
with_gap  = original.copy()
GAP_START, N_MISSING = 20, 10
with_gap[GAP_START: GAP_START + N_MISSING] = np.nan

series_gap = pd.Series(with_gap, index=t)
series_filled = spline_fill_series(series_gap, max_gap=N_MISSING, ctx=5, k=3, s=0.4)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, original,            'k-',  lw=1.5, label='Original')
ax.plot(t, with_gap,            'go',  ms=5,   label='Known points')
ax.plot(t, series_filled.values,'r--', lw=2,   label='Spline fill')
ax.axvspan(GAP_START, GAP_START + N_MISSING - 1, color='yellow', alpha=0.3, label='Gap')
ax.set_title('Spline interpolation test (synthetic data)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'spline_test.png', dpi=110, bbox_inches='tight')
plt.show(); plt.close()

---
## Diagnostic D — Concurrent Completeness (Post-Spline)
**Source:** `data-processing/plotTotalAvailability.py`

More comprehensive availability analysis run **after** spline gap fill:

- **Line plots**: monthly % availability per weather element for each station
- **Scatter overlay**: overall monthly completeness (all required elements present)
- **Concurrent 24H window**: rolling % of time when *all* stations simultaneously
  have a full 24-hour block of data

BELL and MTB are excluded from the concurrent pressure requirement (no `msl_pressure`).

In [ ]:
# SOURCE: data-processing/plotTotalAvailability.py
# Per-station element availability + overall completeness scatter

REQUIRED_FOR_CONCURRENT  = ['air_temp', 'dew_point', 'u_component', 'msl_pressure']
ELEMENTS_FOR_INDIV_PLOTS = ['air_temp', 'dew_point', 'u_component', 'wind_dir_recalc', 'msl_pressure']
DIAG_D_NO_PRES           = ['BELL', 'MTB']
_diag_d_colors = matplotlib.colormaps['tab10'](np.linspace(0, 0.9, len(ELEMENTS_FOR_INDIV_PLOTS)))


def _monthly_pct(series):
    mon = series.groupby(pd.Grouper(freq='ME')).agg(total='count', avail='sum')
    mon['pct'] = np.where(mon['total'] > 0, mon['avail'] / mon['total'] * 100, 0.0)
    return mon


for sid, df in station_data.items():
    is_no_pres = sid in DIAG_D_NO_PRES
    req = [c for c in REQUIRED_FOR_CONCURRENT
           if not (c == 'msl_pressure' and is_no_pres) and c in df.columns]
    df_w = df.copy()
    df_w['_ok'] = df_w[req].notna().all(axis=1).astype(int)
    mon_overall = _monthly_pct(df_w['_ok'])

    fig, ax = plt.subplots(figsize=(15, 5))
    for i, el in enumerate(ELEMENTS_FOR_INDIV_PLOTS):
        if el not in df_w.columns:
            continue
        df_w['_el'] = df_w[el].notna().astype(int)
        mon_el = _monthly_pct(df_w['_el'])
        if not mon_el.empty:
            ax.plot(mon_el.index, mon_el['pct'],
                    label=el.replace('_', ' ').title(),
                    color=_diag_d_colors[i], lw=1.5)

    if not mon_overall.empty:
        ax.scatter(mon_overall.index, mon_overall['pct'],
                   label='Overall Complete', color='black', marker='o', s=30, zorder=5)

    title_suffix = f' (Since {START_YEAR})' if START_YEAR else ''
    ax.set_title(f'Monthly Availability (post-spline) — Station: {sid}{title_suffix}', fontsize=13)
    ax.set_xlabel('Month'); ax.set_ylabel('Percentage Available (%)')
    ax.set_ylim(0, 105); ax.legend(loc='best', fontsize=8)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'{sid}_availability_post_spline.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close(fig)

print('Individual station post-spline availability plots done.')

In [ ]:
# Concurrent 24H rolling completeness — SOURCE: plotTotalAvailability.py

CONCURRENT_WINDOW_STEPS = 48   # 24H at 30-min resolution

concurrent_series = {}
for sid, df in station_data.items():
    is_no_pres = sid in DIAG_D_NO_PRES
    req = [c for c in REQUIRED_FOR_CONCURRENT
           if not (c == 'msl_pressure' and is_no_pres) and c in df.columns]
    if req:
        concurrent_series[sid] = df[req].notna().all(axis=1).astype(int)

if concurrent_series:
    combined_conc = pd.DataFrame(concurrent_series).fillna(0).astype(int)
    all_ok = combined_conc.all(axis=1).astype(int)

    full_idx = pd.date_range(start=all_ok.index.min(), end=all_ok.index.max(), freq='30min')
    all_ok = all_ok.reindex(full_idx, fill_value=0)

    window_ok = (
        all_ok.rolling(window=CONCURRENT_WINDOW_STEPS, min_periods=CONCURRENT_WINDOW_STEPS).sum()
        == CONCURRENT_WINDOW_STEPS
    ).astype(int)

    mon_conc = window_ok.groupby(pd.Grouper(freq='ME')).agg(total='count', good='sum')
    mon_conc['pct'] = np.where(mon_conc['total'] > 0,
                               mon_conc['good'] / mon_conc['total'] * 100, 0.0)

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(mon_conc.index, mon_conc['pct'], marker='o', lw=1.5)
    ax.set_title(
        f'Monthly % of Time with Concurrent 24H Data Completeness\n'
        f'(All {len(concurrent_series)} Stations, Since {START_YEAR})', fontsize=13)
    ax.set_xlabel('Month'); ax.set_ylabel('% of Timesteps with 24H Concurrent Data')
    ax.set_ylim(0, 105)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'concurrent_completeness_24H.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()
    n_conc = int(window_ok.sum())
    print(f'Concurrent 24H complete: {n_conc:,} / {len(window_ok):,} timesteps')
else:
    print('No concurrent data to plot.')

---
## Stage 5 — Drop Columns + Crop to Year Range
**Sources:** `data-processing/drop_columns.py` and `data-processing/crop_to_years.py`

> **Bug fixed from `drop_columns.py`:** that script has two `INPUT_SUBFOLDER_NAME` assignments;
> the second one (`FinalData` → `FinalData`) overrides the first (`SplineInterpolatedData` →
> `FinalData`), so the script reads and writes the same folder.  Here we just drop inline.

> **Bug fixed from `crop_to_years.py`:** `START_YEAR_CONFIG = END_YEAR_CONFIG = 2000` produces
> a single-year output.  `END_YEAR` is now set to 2024 in the config cell.

In [ ]:
# SOURCE: data-processing/drop_columns.py  +  data-processing/crop_to_years.py

for sid, df in station_data.items():
    # drop_columns.py: remove max_gust_speed, aws_flag, wind_dir_recalc
    drop = [c for c in COLUMNS_TO_DROP if c in df.columns]
    df   = df.drop(columns=drop)

    # crop_to_years.py: keep [START_YEAR, END_YEAR] inclusive
    df   = df[(df.index.year >= START_YEAR) & (df.index.year <= END_YEAR)]

    # Refresh completeness flag
    check = [c for c in ['air_temp', 'dew_point', 'msl_pressure', 'wind_speed_recalc']
             if c in df.columns]
    df['data_completeness'] = df[check].notna().all(axis=1).astype(int)

    station_data[sid] = df
    uptime = df['data_completeness'].mean() * 100
    print(f'  {sid}: {len(df):>7,} rows  {START_YEAR}–{END_YEAR}  {uptime:.1f}% complete')

print('Column drop + year crop done.')

---
## Diagnostic C — Missing Timestamp Check
**Source:** `data-processing/find_missing_timestamp.py`

Compare one station's timestamps against the expected 30-min grid and against a
reference station to find any missing rows.

> **Known issue in original:** fully hardcoded (`YSNW`, `YSCN`); all logic runs at module
> level with no `if __name__ == '__main__':` guard.  Here it is wrapped in a function.

In [ ]:
# SOURCE: data-processing/find_missing_timestamp.py

def check_missing_timestamps(station_id, ref_station_id=None, freq='30min'):
    """
    Print timestamps missing from station_id compared with the full expected grid
    and optionally against ref_station_id.
    Source: find_missing_timestamp.py
    """
    if station_id not in station_data:
        print(f'{station_id} not in station_data.'); return

    df1  = station_data[station_id]
    expected = pd.date_range(
        start=f'{START_YEAR}-01-01', end=f'{END_YEAR}-12-31 23:30',
        freq=freq
    )
    missing_vs_grid = sorted(set(expected) - set(df1.index))
    print(f'{station_id} vs expected grid: {len(missing_vs_grid)} missing timestamps')
    for ts in missing_vs_grid[:10]:
        print(f'  {ts}')
    if len(missing_vs_grid) > 10:
        print(f'  ... and {len(missing_vs_grid) - 10} more')

    if ref_station_id and ref_station_id in station_data:
        df2 = station_data[ref_station_id]
        missing_vs_ref = sorted(set(df2.index) - set(df1.index))
        print(f'{station_id} vs {ref_station_id}: {len(missing_vs_ref)} timestamps in ref but not in target')


# ── Run check for YSSY (and compare against first other station if available) ──
sids = list(station_data.keys())
ref  = sids[1] if len(sids) > 1 else None
check_missing_timestamps(TARGET_STATION, ref_station_id=ref)

---
## ★ Parquet Checkpoint — Save All Processed Stations

All data processing is complete.  Save every station to a compressed Parquet file.
From here, `parquet/yssy/` is the single source of truth for the ML stages — you can
reload it without re-running the data processing above.

In [ ]:
for sid, df in station_data.items():
    out = PARQUET_DIR / f'{sid}.parquet'
    df.reset_index().to_parquet(out, index=False)
    print(f'  Saved: {out.name}  ({len(df):,} rows)')

print(f'\nAll processed Parquet files in: {PARQUET_DIR.resolve()}')

In [ ]:
# ── Reload from Parquet (skip here if station_data is still in memory) ─────────
# Uncomment this block when starting a fresh kernel after the processing is done.

# station_data = {}
# for pq in sorted(PARQUET_DIR.glob('*.parquet')):
#     df = pd.read_parquet(pq)
#     df['timestamp'] = pd.to_datetime(df['timestamp'])
#     station_data[pq.stem] = df.set_index('timestamp').sort_index()
# print(f'Reloaded {len(station_data)} stations from Parquet.')

print('station_data is in memory — skipping reload.')

---
## Stage 6 — Feature Engineering & ML Dataset
**Source:** `data-processing/preprocess_data.py`

Each row = one anchor time *t*.  Targets = YSSY U/V at *t*+1 … *t*+48.

| Feature group | Description |
|---|---|
| `YSSY_{feat}_lag0-47` | YSSY all 48 lags (24 hr history) |
| `{SID}_{feat}_lag0-11` | Other stations lags 0–11 (dense, 6 hr) |
| `{SID}_{feat}_lag12,14,...,46` | Other stations every-2nd lag (sparse, 6–23 hr) |
| `PG_{S1}_{S2}_{t,t-6hr,t-12hr,t-24hr}` | Pressure gradients at 4 time points (3 pairs) |
| `PG_..._trend_t_vs_t_minus_{6,12}hr` | Gradient change over 6 hr / 12 hr |
| `{station}_{param}_deriv_t_minus_{a}hr_vs_t_minus_{b}hr` | 3-hr time derivatives (8 × 3-hr periods per station/param) |
| `{station}_{param}_deriv_t_vs_t_minus_{6,12}hr` | Whole-window t vs t-6hr / t-12hr |
| `BELL_v_deriv_eval_at_t_minus_{x}hr_lag_{y}hr` | BELL v-component rate-of-change (12 pts × 2 lags) |
| `sin/cos_time_of_day`, `sin/cos_day_of_year` | Cyclical time (48 half-hours/day, actual leap-year days) |

Stations: `BELL, MTB, YBTH, YCNK, YSBK, YSCN, YSNW, YSRI, YSSY, YSWG`  
BELL and MTB have `msl_pressure` excluded (no pressure sensor).

> **Bug fixed from original:** lines 21–23 of `preprocess_data.py` contain
> `TIMESTEP_MIN` + `UTES = 30` (two broken statements) instead of `TIMESTEP_MINUTES = 30`,
> causing `NameError` on execution.
>
> **Proportions fixed:** original used placeholder 8%/1%/1%; corrected to 80%/10%/10%.
>
> **Implementation:** vectorised `pandas.shift()` instead of the original slow row-by-row loop.

In [ ]:
# SOURCE: data-processing/preprocess_data.py

_PG_PAIRS = [
    ('YBTH', 'YSSY', 'PG_YBTH_YSSY'),
    ('YSSY', 'YCNK', 'PG_YSSY_YCNK'),
    ('YSSY', 'YSNW', 'PG_YSSY_YSNW'),
]
_STEPS_PER_3HR  = 6   # 3 hr / 0.5 hr per step
_NUM_3HR_PERIODS = 8  # 24 hr / 3 hr


def build_ml_dataset(stn_data, target_station, lookback_steps, fcst_steps):
    """
    Vectorised feature engineering matching preprocess_data.py.

    A) Lagged raw values  (target: lags 0-47; others: 0-11 dense + 12,14,...,46 sparse)
    B) Cyclical time  (sin/cos of half-hour-of-day and actual day-of-year)
    C) Pressure gradients at t, t-6hr, t-12hr, t-24hr + trend vs 6hr/12hr
    D) 3-hr time derivatives + t vs t-6hr / t vs t-12hr per station/param
    E) BELL v-component rate-of-change (12 eval pts x 2 lags)
    """
    common_idx = stn_data[target_station].index
    for sid in STATIONS:
        if sid in stn_data:
            common_idx = common_idx.intersection(stn_data[sid].index)
    common_idx = common_idx.sort_values()
    print(f'Common index: {common_idx[0]} -> {common_idx[-1]}  ({len(common_idx):,} steps)')

    aligned = {sid: stn_data[sid].reindex(common_idx) for sid in STATIONS if sid in stn_data}
    feat_dict = {}

    # ── A. Raw lag features ────────────────────────────────────────────────
    for sid, df in aligned.items():
        feats = [f for f in ALL_FILE_FEATURES
                 if not (f == 'msl_pressure' and sid in STATIONS_NO_PRESSURE)
                 and f in df.columns]
        lags = (range(lookback_steps) if sid == target_station
                else list(range(12)) + list(range(12, lookback_steps, 2)))
        for lag in lags:
            for col in feats:
                feat_dict[f'{sid}_{col}_lag{lag}'] = df[col].shift(lag)

    # ── B. Cyclical time features ──────────────────────────────────────────
    half_hrs = common_idx.hour * 2 + common_idx.minute // 30
    feat_dict['sin_time_of_day'] = np.sin(2 * np.pi * half_hrs / 48.0)
    feat_dict['cos_time_of_day'] = np.cos(2 * np.pi * half_hrs / 48.0)
    days_in_yr = np.where(common_idx.is_leap_year, 366, 365)
    feat_dict['sin_day_of_year'] = np.sin(2 * np.pi * common_idx.dayofyear / days_in_yr)
    feat_dict['cos_day_of_year'] = np.cos(2 * np.pi * common_idx.dayofyear / days_in_yr)

    # ── C. Pressure gradients ──────────────────────────────────────────────
    _pg_shifts = {'t': 0, 't_minus_6hr': 12, 't_minus_12hr': 24, 't_minus_24hr': 48}
    for s1, s2, label in _PG_PAIRS:
        grads = {}
        for time_lbl, shift in _pg_shifts.items():
            grad = aligned[s1]['msl_pressure'].shift(shift) - aligned[s2]['msl_pressure'].shift(shift)
            feat_dict[f'{label}_{time_lbl}'] = grad
            grads[time_lbl] = grad
        feat_dict[f'{label}_trend_t_vs_t_minus_6hr']  = grads['t'] - grads['t_minus_6hr']
        feat_dict[f'{label}_trend_t_vs_t_minus_12hr'] = grads['t'] - grads['t_minus_12hr']

    # ── D. Time derivatives ────────────────────────────────────────────────
    for sid, df in aligned.items():
        for param in ['air_temp', 'dew_point', 'msl_pressure']:
            if param not in df.columns:
                continue
            if param == 'msl_pressure' and sid in STATIONS_NO_PRESSURE:
                continue
            s = df[param]
            for period in range(_NUM_3HR_PERIODS):   # 8 x 3-hr periods over 24-hr lookback
                end_steps, start_steps = period * _STEPS_PER_3HR, (period + 1) * _STEPS_PER_3HR
                end_hr, start_hr = period * 3, (period + 1) * 3
                feat_dict[f'{sid}_{param}_deriv_t_minus_{end_hr}hr_vs_t_minus_{start_hr}hr'] = (
                    s.shift(end_steps) - s.shift(start_steps)
                )
            feat_dict[f'{sid}_{param}_deriv_t_vs_t_minus_6hr']  = s - s.shift(12)
            feat_dict[f'{sid}_{param}_deriv_t_vs_t_minus_12hr'] = s - s.shift(24)

    # ── E. BELL v-component derivatives ───────────────────────────────────
    if 'BELL' in aligned and 'v_component' in aligned['BELL'].columns:
        bell_v = aligned['BELL']['v_component']
        for step_offset in range(12):
            lbl = f'{step_offset * 0.5:.1f}'
            feat_dict[f'BELL_v_deriv_eval_at_t_minus_{lbl}hr_lag_0.5hr'] = (
                bell_v.shift(step_offset) - bell_v.shift(step_offset + 1)
            )
            feat_dict[f'BELL_v_deriv_eval_at_t_minus_{lbl}hr_lag_1.0hr'] = (
                bell_v.shift(step_offset) - bell_v.shift(step_offset + 2)
            )

    features_df = pd.DataFrame(feat_dict, index=common_idx)

    # ── Targets ────────────────────────────────────────────────────────────
    tgt = aligned[target_station]
    tgt_dict = {}
    for h in range(1, fcst_steps + 1):
        tgt_dict[f'{target_station}_u_forecast_t_plus_{h}'] = tgt['u_component'].shift(-h)
        tgt_dict[f'{target_station}_v_forecast_t_plus_{h}'] = tgt['v_component'].shift(-h)
    targets_df = pd.DataFrame(tgt_dict, index=common_idx)

    ts_col = pd.Series(common_idx, index=common_idx, name=TIMESTAMP_COL)
    ml_df  = pd.concat([ts_col, features_df, targets_df], axis=1).dropna()
    print(f'ML samples: {len(ml_df):,}  |  columns: {ml_df.shape[1]}')
    return ml_df


print('Building ML dataset (vectorised, matching preprocess_data.py)...')
ml_df = build_ml_dataset(station_data, TARGET_STATION, LOOKBACK_STEPS, FORECAST_STEPS)

In [ ]:
# ── Chronological 80/10/10 split ────────────────────────────────────────────
# SOURCE: preprocess_data.py  (original had 8%/1%/1% placeholder — fixed to 80/10/10)

ml_df   = ml_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
n       = len(ml_df)
n_train = int(n * TRAIN_PROP)
n_val   = int(n * VAL_PROP)

splits = {
    'training':   ml_df.iloc[:n_train],
    'validation': ml_df.iloc[n_train: n_train + n_val],
    'test':       ml_df.iloc[n_train + n_val:],
}

ML_PARQUET_DIR = PARQUET_DIR / 'ml'
ML_PARQUET_DIR.mkdir(exist_ok=True)

for name, df in splits.items():
    out = ML_PARQUET_DIR / f'{name}_dataset.parquet'
    df.to_parquet(out, index=False)
    t0 = pd.to_datetime(df[TIMESTAMP_COL].iloc[0]).strftime('%Y-%m-%d')
    t1 = pd.to_datetime(df[TIMESTAMP_COL].iloc[-1]).strftime('%Y-%m-%d')
    print(f'  {name:10s}: {len(df):>6,} samples  [{t0} → {t1}]')

---
## Stage 7A — Quick Model (fixed hyperparameters)
**Source:** `model/YSSY_winds_24hr.py`

Fastest path to a working model: single `MultiOutputRegressor` with fixed LightGBM params,
optionally subsampled training data.  Good for quickly checking the feature pipeline.

In [ ]:
# SOURCE: model/YSSY_winds_24hr.py

def load_ml_parquet(path):
    """Load a ML dataset Parquet; return (full_df, X, y, timestamps, target_cols, feature_cols)."""
    df = pd.read_parquet(path)
    target_cols = sorted(
        [c for c in df.columns
         if c.startswith(f'{TARGET_STATION}_u_forecast_t_plus_') or
            c.startswith(f'{TARGET_STATION}_v_forecast_t_plus_')],
        key=lambda x: (int(x.rsplit('_', 1)[-1]), x.split('_')[1])
    )
    feat_cols = [c for c in df.columns if c != TIMESTAMP_COL and c not in target_cols]
    return df, df[feat_cols], df[target_cols], pd.to_datetime(df[TIMESTAMP_COL]), target_cols, feat_cols


train_full, X_train, y_train, ts_train, TARGET_COLS, FEATURE_COLS = load_ml_parquet(
    ML_PARQUET_DIR / 'training_dataset.parquet'
)
val_full,   X_val,   y_val,   ts_val,   _,           _            = load_ml_parquet(
    ML_PARQUET_DIR / 'validation_dataset.parquet'
)
test_full,  X_test,  y_test,  ts_test,  _,           _            = load_ml_parquet(
    ML_PARQUET_DIR / 'test_dataset.parquet'
)

# Quick-model fixed params (from YSSY_winds_24hr.py FIXED_LGBM_PARAMS)
QUICK_PARAMS = {
    'objective': 'regression_l2', 'metric': 'l2',
    'n_estimators': 10, 'learning_rate': 0.05, 'num_leaves': 10,
    'max_depth': 10, 'min_child_samples': 10, 'subsample': 0.8,
    'colsample_bytree': 0.7, 'reg_alpha': 0.5, 'reg_lambda': 0.1,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1,
}

# Subsample for speed (YSSY_winds_24hr.py: TRAIN_DATA_SUBSAMPLE_FRACTION = 0.1)
SUBSAMPLE = 0.1
X_tr_q = X_train.sample(frac=SUBSAMPLE, random_state=42)
y_tr_q = y_train.loc[X_tr_q.index]

model_quick = MultiOutputRegressor(lgb.LGBMRegressor(**QUICK_PARAMS))
t0 = time.time()
model_quick.fit(X_tr_q, y_tr_q)
print(f'Quick model trained in {time.time()-t0:.1f}s  ({X_tr_q.shape[0]:,} samples, {y_tr_q.shape[1]} targets)')

joblib.dump(model_quick, MODEL_DIR / 'yssy_quick_model.joblib')
print('Saved: models/yssy_quick_model.joblib')

---
## Stage 7B — Full Multi-output Model (tuned hyperparameters)
**Source:** `model/YSSY_LightGMB.py`

Single `MultiOutputRegressor` using the Optuna-tuned hyperparameters from `YSSY_LightGMB.py`.
Trains on the full training set.

In [ ]:
# SOURCE: model/YSSY_LightGMB.py → train_final_lgbm_model()

# LGBM_PARAMS comes from the config cell (tuned values from YSSY_LightGMB.py)
model = MultiOutputRegressor(lgb.LGBMRegressor(**LGBM_PARAMS))

print(f'Training on {X_train.shape[0]:,} samples × {X_train.shape[1]} features → {y_train.shape[1]} targets')
print('(This may take several minutes.)')
t0 = time.time()
model.fit(X_train, y_train)
print(f'Done in {time.time()-t0:.0f}s.')

joblib.dump(model, MODEL_DIR / 'yssy_wind_model.joblib')
print('Saved: models/yssy_wind_model.joblib')

---
## Stage 7C — [Optional] Per-target Individual Models
**Sources:** `model/tune_single_model.py` (Optuna HP search) + `model/train_final_individual.py` (training)

`train_final_individual.py` trains **one LightGBM per forecast target** (96 models total:
U and V for each of 48 steps).  Each model uses early stopping on the validation set.
This is slower but allows per-step tuning and matches the `evaluate_all_24hr_models.py` loader.

`tune_single_model.py` runs an Optuna study for a single target column; the resulting
hyperparameters JSON feeds `train_final_individual.py`.

In [ ]:
# SOURCE: model/train_final_individual.py
# Uncomment to run. Training 96 models takes 30–60 min on full data.

# INDIV_PARAMS = {  # from train_final_individual.py CHOSEN_HYPERPARAMETERS (second assignment)
#     'learning_rate': 0.01, 'num_leaves': 300, 'max_depth': 40,
#     'min_child_samples': 100, 'subsample': 0.7, 'colsample_bytree': 0.7,
#     'reg_alpha': 0.5, 'reg_lambda': 0.1, 'n_estimators': 4000,
#     'objective': 'regression_l2', 'metric': 'l2',
#     'random_state': 42, 'n_jobs': -1, 'verbose': -1,
# }

# INDIV_DIR = MODEL_DIR / 'individual_models'
# INDIV_DIR.mkdir(exist_ok=True)

# for target_col in TARGET_COLS:
#     lgbm_single = lgb.LGBMRegressor(**INDIV_PARAMS)
#     lgbm_single.fit(
#         X_train, y_train[target_col],
#         eval_set=[(X_val, y_val[target_col])],
#         callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
#     )
#     joblib.dump(lgbm_single, INDIV_DIR / f'{target_col}.joblib')
#     print(f'  Saved {target_col}  (best iter={lgbm_single.best_iteration_})')

print('Per-target individual model block is commented out.')
print('Uncomment and run if you want individual .joblib per forecast step.')

---
## Stage 8 — Evaluation
**Source:** `model/evaluate_all_24hr_models.py`

Compute MAE and MSE on the test set.  Plot error vs forecast horizon.

In [ ]:
# SOURCE: model/evaluate_all_24hr_models.py → make_predictions_with_all_models()

preds = model.predict(X_test)   # shape: (n_test, n_targets)

overall_mae = mean_absolute_error(y_test, preds)
overall_mse = mean_squared_error(y_test,  preds)
print(f'Overall Test MAE : {overall_mae:.4f} kt')
print(f'Overall Test MSE : {overall_mse:.4f} kt²')

# Per-horizon metrics
mae_u, mae_v = [], []
for step in range(1, FORECAST_STEPS + 1):
    iu = TARGET_COLS.index(f'{TARGET_STATION}_u_forecast_t_plus_{step}')
    iv = TARGET_COLS.index(f'{TARGET_STATION}_v_forecast_t_plus_{step}')
    mae_u.append(mean_absolute_error(y_test.iloc[:, iu], preds[:, iu]))
    mae_v.append(mean_absolute_error(y_test.iloc[:, iv], preds[:, iv]))

hours = [h * 0.5 for h in range(1, FORECAST_STEPS + 1)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hours, mae_u, 'b-o', ms=3, lw=1.5, label='U component MAE')
ax.plot(hours, mae_v, 'r-o', ms=3, lw=1.5, label='V component MAE')
ax.set_xlabel('Forecast horizon (hours)'); ax.set_ylabel('MAE (knots)')
ax.set_title('Test Set — Forecast Error vs Horizon')
ax.legend(); ax.grid(True, alpha=0.4)
ax.set_xlim(0, 24); ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'mae_vs_horizon.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('Saved: plots/yssy/mae_vs_horizon.png')

In [ ]:
# Summary table — mimics evaluate_all_24hr_models.py console output
rows = [(f'{h:.0f} hr', mae_u[h*2-1], mae_v[h*2-1]) for h in [1, 3, 6, 12, 24]]
summary = pd.DataFrame(rows, columns=['Horizon', 'U MAE (kt)', 'V MAE (kt)'])
summary['Mean MAE (kt)'] = (summary['U MAE (kt)'] + summary['V MAE (kt)']) / 2
display(summary.set_index('Horizon').round(4))

---
## Stage 9 — Visual Display
**Sources:** `model/plot_samples_24hr.py` (individual models, Jan-2024 loop) and `model/plot_test_samples.py` (quick model, debug version)

For each sample anchor time *t*, plot:
- **Gray** — past observed YSSY wind (from lag features)
- **Blue** — actual future YSSY wind (from target columns)
- **Red dashed** — model forecast

Upper panel: wind speed (kt). Lower panel: wind direction (°).

In [ ]:
# SOURCE: model/plot_samples_24hr.py → plot_forecast_vs_actual()
#         model/plot_test_samples.py  → plot_forecast_vs_actual()

def plot_forecast_sample(anchor_ts, mdl, test_row, target_cols, feature_cols, plot_idx=''):
    """
    Wind forecast plot: past observed + actual future + model forecast.
    Source: plot_samples_24hr.py → plot_forecast_vs_actual()
    """
    # Past YSSY winds from lag features (lag0 = t, lag11 = t-5.5hr)
    u_lags = sorted([c for c in feature_cols if c.startswith(f'{TARGET_STATION}_u_component_lag')],
                    key=lambda x: int(x.split('lag')[-1]))
    v_lags = sorted([c for c in feature_cols if c.startswith(f'{TARGET_STATION}_v_component_lag')],
                    key=lambda x: int(x.split('lag')[-1]))

    past_u   = test_row[u_lags].values[::-1]   # oldest → newest
    past_v   = test_row[v_lags].values[::-1]
    n_past   = len(u_lags)
    past_t   = [anchor_ts - timedelta(minutes=30*(n_past-1-i)) for i in range(n_past)]
    future_t = [anchor_ts + timedelta(minutes=30*h) for h in range(1, FORECAST_STEPS+1)]

    actual_u = np.array([test_row.get(f'{TARGET_STATION}_u_forecast_t_plus_{h}', np.nan)
                         for h in range(1, FORECAST_STEPS+1)])
    actual_v = np.array([test_row.get(f'{TARGET_STATION}_v_forecast_t_plus_{h}', np.nan)
                         for h in range(1, FORECAST_STEPS+1)])

    X_smp    = pd.DataFrame([test_row[feature_cols].values], columns=feature_cols)
    pred     = mdl.predict(X_smp)[0]
    pred_u   = np.array([pred[target_cols.index(f'{TARGET_STATION}_u_forecast_t_plus_{h}')]
                         for h in range(1, FORECAST_STEPS+1)])
    pred_v   = np.array([pred[target_cols.index(f'{TARGET_STATION}_v_forecast_t_plus_{h}')]
                         for h in range(1, FORECAST_STEPS+1)])

    past_spd,   past_dir   = uv_to_wind(past_u,   past_v)
    actual_spd, actual_dir = uv_to_wind(actual_u, actual_v)
    pred_spd,   pred_dir   = uv_to_wind(pred_u,   pred_v)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    fig.suptitle(f'YSSY Wind Forecast — anchor: {anchor_ts.strftime("%Y-%m-%d %H:%M UTC")}', fontsize=13)

    ax1.plot(past_t,   past_spd,   'o-',  color='dimgray',   lw=1.5, ms=4, label='Observed (input)')
    ax1.plot(future_t, actual_spd, 's-',  color='steelblue', lw=1.5, ms=4, label='Observed (actual future)')
    ax1.plot(future_t, pred_spd,   'x--', color='crimson',   lw=2.0, ms=5, label='Forecast')
    ax1.axvline(anchor_ts, color='k', ls=':', lw=1, alpha=0.5)
    ax1.set_ylabel('Wind speed (kt)'); ax1.legend(loc='upper left', fontsize=8)
    ax1.set_ylim(bottom=0); ax1.grid(True, alpha=0.3)

    ax2.scatter(past_t,   past_dir,   color='dimgray',   s=18, label='Observed (input)')
    ax2.scatter(future_t, actual_dir, color='steelblue', s=18, label='Observed (future)')
    ax2.scatter(future_t, pred_dir,   color='crimson',   s=30, marker='x', label='Forecast')
    ax2.axvline(anchor_ts, color='k', ls=':', lw=1, alpha=0.5)
    ax2.set_ylabel('Wind dir (°)'); ax2.set_ylim(0, 360)
    ax2.set_yticks([0, 90, 180, 270, 360]); ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)

    ax2.set_xlabel('Time')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d-%b'))
    ax2.xaxis.set_major_locator(mdates.HourLocator(interval=3))

    plt.tight_layout()
    fname = PLOTS_DIR / f'forecast_{anchor_ts.strftime("%Y%m%d_%H%M")}{plot_idx}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname.name}')


print('Forecast plot function ready.')

In [ ]:
# SOURCE: model/plot_samples_24hr.py → random sample loop

N_SAMPLES = 5
random.seed(42)
indices = random.sample(range(len(test_full)), min(N_SAMPLES, len(test_full)))

for i, idx in enumerate(indices):
    row = test_full.iloc[idx]
    anchor_ts = pd.to_datetime(row[TIMESTAMP_COL])
    print(f'\n--- Sample {i+1}/{N_SAMPLES}:  {anchor_ts} ---')
    plot_forecast_sample(anchor_ts, model, row, TARGET_COLS, FEATURE_COLS, plot_idx=f'_s{i+1}')

print(f'\nPlots saved to: {PLOTS_DIR.resolve()}')

In [ ]:
# SOURCE: model/plot_samples_24hr.py → loop over January 2024 (or any date range)
# SOURCE: model/plot_test_samples.py → manual timestamp plot

LOOP_START = '2024-01-01 00:00'
LOOP_END   = '2024-01-02 00:00'   # narrow range for demo; extend as needed

time_range = pd.date_range(start=LOOP_START, end=LOOP_END, freq='6h')  # every 6 hr
ts_index   = pd.to_datetime(test_full[TIMESTAMP_COL])

for ts in time_range:
    match = test_full[ts_index == ts]
    if match.empty:
        continue
    plot_forecast_sample(ts, model, match.iloc[0], TARGET_COLS, FEATURE_COLS,
                         plot_idx=f'_{ts.strftime("%Y%m%d_%H%M")}')

print('Date-range loop done.')